# Contrastive representation learning for Roman Urdu hate speech (RUHSOLD)

Extends the MPhil thesis *Hate Speech Detection in Roman Urdu Text* with contrastive fine-tuning of `bert-base-multilingual-cased`:

| Method | Loss | Supervision |
|---|---|---|
| `simcse` | MultipleNegativesRankingLoss on (x, x) — dropout as augmentation | none |
| `contrastive` | OnlineContrastiveLoss (margin-based, Hadsell et al. 2006) | same/different-class pairs |
| `mnrl` | MultipleNegativesRankingLoss with explicit cross-class hard negative | (anchor, positive, negative) triplets |

Each embedding space is evaluated with a frozen linear probe (macro-F1, MCC, ROC-AUC, FNR, FPR), a cost-sensitive probe that penalises missed hate speech, kNN accuracy, silhouette, and alignment/uniformity (Wang & Isola, 2020).

**Runtime → Change runtime type → T4 GPU** before running. Full run takes roughly 15–25 minutes.

In [ ]:
!git clone https://github.com/quratulain77-saeed/Hate-Speech-detection-in-text.git
%cd Hate-Speech-detection-in-text/contrastive
!pip install -q -r requirements.txt

In [ ]:
import torch; print("GPU:", torch.cuda.is_available())

## 1. Baseline: pre-trained mBERT embeddings (no contrastive training)

In [ ]:
!python src/evaluate.py --model bert-base-multilingual-cased --name baseline --out results

## 2. Unsupervised SimCSE

In [ ]:
!python src/train.py --method simcse --epochs 1 --batch_size 64 --out models/simcse
!python src/evaluate.py --model models/simcse --name simcse --out results

## 3. Supervised margin-based contrastive loss

In [ ]:
!python src/train.py --method contrastive --epochs 2 --batch_size 32 --out models/contrastive
!python src/evaluate.py --model models/contrastive --name contrastive --out results

## 4. Supervised MNRL with hard negatives

In [ ]:
!python src/train.py --method mnrl --epochs 2 --batch_size 32 --out models/mnrl
!python src/evaluate.py --model models/mnrl --name mnrl --out results

## 5. Results table and t-SNE

In [ ]:
!python src/report.py --results results
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results_table.md").read()))
display(Image("results/tsne.png"))

## 6. Save results
Download the `results/` folder (JSON files, `results_table.md`, `tsne.png`) and commit it to the repo so the README can show your real numbers.
Model folders are large (~700 MB each) — don't commit them.

In [ ]:
!zip -r results.zip results
from google.colab import files; files.download("results.zip")